# 추론시간과 처리율

CPU에서 FP32 모델을 실행하고, 연산량을 걸린 시간으로 나누어 처리율을 구합니다.

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
import embedded_lab as lab
from embedded_lab import table

lab.start()

## 1. 추론시간 측정
모델과 입력을 준비한 뒤, 준비 실행을 거쳐 **최소 2초·30회** 반복합니다.
모델 호출만 측정하며 파일 읽기와 입력 변환은 제외합니다. 호출시간은 반복 측정의 중앙값으로 표시합니다.

In [ ]:
DEVICE_LABEL = '내 노트북 CPU'
CONDITIONS = '전원·냉각·다른 프로그램 상태를 적으세요'
SIZE, BATCH = 48, 1
x = lab.inputs(SIZE, BATCH)
actual_batch = int(x.shape[0])
ops_batch = sum(r['ops'] for r in lab.count_ops(lab.load(), x))
model = lab.load()
measurement = lab.benchmark(model, x, min_seconds=2.0, min_repeats=30)
table(['호출시간 ms', '반복 횟수', '전체 경과시간 s'],
      [[round(measurement['median_s'] * 1000, 3), measurement['repeat_count'],
        round(measurement['wall_elapsed_s'], 3)]])

## 2. 처리율
**처리율 = 처리한 이미지 수 또는 연산 수 ÷ 전체 경과시간**입니다. 반복문의 작은 실행 비용도 경과시간에 포함됩니다.

| 단위 | 의미 |
|---|---|
| images/s | 초당 처리한 이미지 수 |
| GOP/s | 초당 10⁹ operations |
| GFLOPS | 초당 10⁹ 부동소수점 연산 |
| TOPS | 초당 10¹² operations |

이번 FP32 계산에서는 **GFLOPS = GOP/s**, **TOPS = GOP/s ÷ 1,000**입니다.
Conv/Linear의 연산량을 모델 전체 시간으로 나눈 **이 모델의 실효 처리율**이며, 장치 최대 성능과는 다릅니다.

### ✏️ C01. 측정값으로 계산
반복 횟수를 N, 전체 시간을 T라고 할 때 아래 값을 dict로 반환하세요.

| 키 | 계산 |
|---|---|
| `images_s` | N × 실제 배치 ÷ T |
| `gops` | N × 배치당 ops ÷ T ÷ 10⁹ |
| `gflops_fp32` | gops |
| `tops` | gops ÷ 1,000 |

In [ ]:
def rates(ops_batch, measurement, actual_batch):
    n = measurement['repeat_count']
    elapsed = measurement['wall_elapsed_s']
    raise NotImplementedError('C01 함수를 완성하세요.')

In [ ]:
probe = {'repeat_count': 4, 'wall_elapsed_s': 0.05}
check = rates(200_000_000, probe, 2)
assert np.isclose(check['images_s'], 160)
assert np.isclose(check['gops'], 16) and np.isclose(check['tops'], 0.016)
assert check['gflops_fp32'] == check['gops']
result = rates(ops_batch, measurement, actual_batch)
table(['처리량', 'GOP/s', '실효 GFLOPS', 'TOPS'],
      [[f"{result['images_s']:.2f} images/s", f"{result['gops']:.6f}",
        f"{result['gflops_fp32']:.6f}", f"{result['tops']:.9f}"]])
print('C01 완료')

## 3. 응답시간과 처리량
배치 4가 30ms에 완료되면 약 **133장/s**를 처리합니다. 각 이미지는 결과가 나올 때까지 **30ms**를 기다립니다.
30÷4=7.5ms는 개별 이미지의 응답시간이 아닙니다.

**확인:** 이번에 구한 TOPS를 장치의 최대 TOPS로 볼 수 있나요?

**답:**

## 4. 저장
측정 결과를 `results` 폴더에 저장합니다. 장치 비교에서 이 파일을 사용합니다.

In [ ]:
record = dict(schema_version=2, label=DEVICE_LABEL, conditions=CONDITIONS,
              environment=lab.environment(model, x), size=SIZE, batch=actual_batch,
              ops_batch=ops_batch, measurement=measurement, rates=result)
saved = lab.save_record(record)
print('저장:', saved.name)